# ReviewLens — Llama 3.1 8B fine-tuned with QLoRA (JN)

Fine-tunes `Llama-3.1-8B-Instruct` (4-bit base, LoRA adapters via PEFT) to generate one
sentiment label, then evaluates on the shared test set and on the 600 AI-labeled reviews (Experiment 2).

Same prompt and same label-extraction rule as Dinh's prompted Llama, so the two Llama methods are comparable.

**Before running**
1. Runtime → Change runtime type → GPU (A100 recommended; L4 works but training is several hours).
2. Accept the Llama 3.1 licence on Hugging Face and add your token as a Colab secret named `HF_TOKEN`.
3. Dinh's `ReviewLens/` folder must be visible at `MyDrive/ReviewLens` (right-click the shared folder → *Add shortcut to Drive*).

Everything is written to `ReviewLens/jn_llama_qlora/` on Drive. If Colab disconnects, just run all cells again:
training resumes from the last checkpoint, and a finished adapter is reloaded instead of retrained.

In [ ]:
!pip -q install -U bitsandbytes peft

In [ ]:
import json, re, time
from pathlib import Path
import pandas as pd, torch
from google.colab import drive
from huggingface_hub import login, get_token

drive.mount('/content/drive')
if not get_token(): login()

ROOT = Path('/content/drive/MyDrive/ReviewLens')
DATA, OUT = ROOT / 'shared_data', ROOT / 'jn_llama_qlora'
OUT.mkdir(exist_ok=True)

MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
LABELS = ['Negative', 'Neutral', 'Positive']
SEED = 42
TRAIN_N = None        # None = all 40,000 training reviews; set e.g. 12000 for a faster run (report it!)
EPOCHS = 1
LR = 2e-4
TRAIN_BS = 16         # lower to 8 if you hit out-of-memory
GEN_BS = 32
MAX_NEW_TOKENS = 8

GPU = torch.cuda.get_device_name(0)
print(GPU)

## 1. Data (shared split — unchanged)

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
val = pd.read_csv(DATA / 'val.csv')
test = pd.read_csv(DATA / 'test.csv')
exp2 = pd.read_csv(DATA / 'ai_labeled_600.csv')

if TRAIN_N:
    train = train.groupby('label').sample(TRAIN_N // 3, random_state=SEED)
train = train.sample(frac=1, random_state=SEED)
# 200 per class from validation: used for eval loss during training and a sanity check before test
val_small = val.groupby('label').sample(200, random_state=SEED)
print(len(train), len(val_small), len(test), len(exp2))

## 2. Label extraction and metrics
Same rule as the prompted method: exact match after normalising, otherwise the first
whole-word, case-insensitive label found in the output, otherwise `Invalid`.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

def extract_label(raw):
    norm = raw.strip().lower()
    for l in LABELS:
        if norm == l.lower():
            return l, 'exact'
    m = re.search(r'\b(' + '|'.join(LABELS) + r')\b', raw, re.I)
    return (m.group(1).capitalize(), 'fallback') if m else ('Invalid', 'invalid')

def metrics_row(setting, gold, pred):
    gold, pred = list(gold), list(pred)
    row = {'setting': setting, 'n': len(gold), 'accuracy': accuracy_score(gold, pred)}
    for avg in ('macro', 'micro'):
        p, r, f, _ = precision_recall_fscore_support(gold, pred, labels=LABELS, average=avg, zero_division=0)
        row |= {f'{avg}_precision': p, f'{avg}_recall': r, f'{avg}_f1': f}
    row['invalid_rate'] = sum(p == 'Invalid' for p in pred) / len(pred)
    p, r, f, s = precision_recall_fscore_support(gold, pred, labels=LABELS, zero_division=0)
    for i, l in enumerate(LABELS):
        row |= {f'{l}_precision': p[i], f'{l}_recall': r[i], f'{l}_f1': f[i], f'{l}_support': int(s[i])}
    return row

assert extract_label(' negative\n') == ('Negative', 'exact')
assert extract_label('Neutral\n\nThe review mentions both positive and') == ('Neutral', 'fallback')
assert extract_label('I think it is POSITIVE.') == ('Positive', 'fallback')
assert extract_label('mixed') == ('Invalid', 'invalid')

## 3. Model: 4-bit base + LoRA adapters

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = '<|finetune_right_pad_id|>'

base = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map='auto',
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16))
base.generation_config.pad_token_id = tok.pad_token_id

# Identical to the prompted method's zero-shot prompt
SYSTEM = ('You are a sentiment classifier for e-commerce product reviews. Read the customer review and '
          'classify its overall sentiment as Positive, Neutral, or Negative. '
          'Return only one label: Positive, Neutral, or Negative.')

def build_prompt(text):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': f'Review:\n"""\n{text}\n"""\n\nReturn only one label: Positive, Neutral, or Negative.'}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# If Dinh's saved prompt is on Drive, check ours is character-for-character the same
ref = ROOT / 'quoc_llama_prompting' / 'prompt_example_zero_shot.txt'
if ref.exists():
    assert build_prompt(test.text[0]).strip() == ref.read_text().strip(), 'prompt differs from the prompted method'
    print('prompt matches the prompted method')

## 4. QLoRA training
The target is only the label (`Positive` / `Neutral` / `Negative`) followed by the end-of-turn token;
the prompt tokens are masked out of the loss.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

ADAPTER = OUT / 'adapter'
CKPT = OUT / 'checkpoints'

def encode(row):
    p = tok(build_prompt(row['text']), add_special_tokens=False)['input_ids']
    a = tok(row['label'] + '<|eot_id|>', add_special_tokens=False)['input_ids']
    return {'input_ids': p + a, 'attention_mask': [1] * len(p + a), 'labels': [-100] * len(p) + a}

def to_ds(df):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(encode, remove_columns=['text', 'label'])

if ADAPTER.exists():
    model = PeftModel.from_pretrained(base, ADAPTER)
    train_info = json.loads((OUT / 'train_info.json').read_text())
    print('loaded trained adapter, skipping training')
else:
    model = get_peft_model(
        prepare_model_for_kbit_training(base),
        LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                   target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    model.print_trainable_parameters()
    tok.padding_side = 'right'
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(CKPT), num_train_epochs=EPOCHS, learning_rate=LR,
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=TRAIN_BS,
            lr_scheduler_type='cosine', warmup_steps=100, optim='paged_adamw_8bit', bf16=True,
            logging_steps=25, eval_strategy='steps', eval_steps=500, save_steps=500, save_total_limit=2,
            label_names=['labels'], report_to='none', seed=SEED),
        train_dataset=to_ds(train), eval_dataset=to_ds(val_small),
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    resumed = any(CKPT.glob('checkpoint-*'))
    out = trainer.train(resume_from_checkpoint=resumed)
    model.save_pretrained(ADAPTER)
    train_info = {
        'gpu': GPU, 'train_n': len(train), 'epochs': EPOCHS, 'lr': LR, 'batch_size': TRAIN_BS,
        'train_minutes': out.metrics['train_runtime'] / 60,
        # shortcut: after a resume train_runtime only covers the resumed part; rerun uninterrupted for a clean number
        'resumed_from_checkpoint': resumed,
        'log': trainer.state.log_history}
    (OUT / 'train_info.json').write_text(json.dumps(train_info, indent=1))
print(f"training: {train_info['train_minutes']:.1f} min on {train_info['gpu']} (resumed={train_info['resumed_from_checkpoint']})")

## 5. Inference

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm.auto import tqdm

@torch.no_grad()
def generate(texts):
    model.eval()
    tok.padding_side = 'left'
    raws = []
    for i in tqdm(range(0, len(texts), GEN_BS)):
        enc = tok([build_prompt(t) for t in texts[i:i + GEN_BS]], return_tensors='pt',
                  padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        raws += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return raws

timings = {}

def predict(df, name):
    t = time.time()
    raw = generate(df.text.tolist())
    timings[name] = (time.time() - t) / len(df)
    ex = [extract_label(r) for r in raw]
    preds = pd.DataFrame({'review_id': df.review_id.values, 'raw_output': raw,
                          'pred': [e[0] for e in ex], 'extract_method': [e[1] for e in ex]})
    preds.to_csv(OUT / f'preds_{name}.csv', index=False)
    print(name, preds.extract_method.value_counts().to_dict())
    return preds.pred.values

def save_cm(name, gold, pred, gold_name):
    cm = confusion_matrix(gold, pred, labels=LABELS)
    pd.DataFrame(cm, index=pd.Index(LABELS, name=gold_name), columns=LABELS).to_csv(OUT / f'cm_{name}.csv')
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap='Blues', colorbar=False)
    plt.ylabel(gold_name); plt.xlabel('QLoRA prediction'); plt.title(name)
    plt.savefig(OUT / f'cm_{name}.png', dpi=150, bbox_inches='tight'); plt.show()

# Sanity check on validation before touching the test set
val_pred = predict(val_small, 'val')
val_row = metrics_row('QLoRA — val (600)', val_small.label, val_pred)
print(f"val accuracy {val_row['accuracy']:.4f}  macro-F1 {val_row['macro_f1']:.4f}")

## 6. Test set and Experiment 2

In [ ]:
test_pred = predict(test, 'test')
exp2_pred = predict(exp2, 'exp2')

rows = [
    val_row,
    metrics_row('QLoRA — test', test.label, test_pred),
    metrics_row('QLoRA — Exp2 vs AI labels', exp2.ai_label, exp2_pred),
    metrics_row('QLoRA — Exp2 vs star labels (same 600)', exp2.star_label, exp2_pred),
]
results = pd.DataFrame(rows)
results.to_csv(OUT / 'results_summary.csv', index=False)

save_cm('test', test.label, test_pred, 'Star label')
save_cm('exp2_vs_ai', exp2.ai_label, exp2_pred, 'AI label')
save_cm('exp2_vs_stars', exp2.star_label, exp2_pred, 'Star label')

cost = pd.DataFrame([{
    'setting': 'QLoRA', 'gpu_inference': GPU, 'batch_size': GEN_BS, 'timed_reviews': len(test),
    'sec_per_review': timings['test'], 'reviews_per_sec': 1 / timings['test'],
    'gpu_training': train_info['gpu'], 'training_minutes': train_info['train_minutes'],
    'train_n': train_info['train_n'], 'epochs': train_info['epochs']}])
cost.to_csv(OUT / 'inference_cost.csv', index=False)

results.set_index('setting').T

## 7. Summary to send to Dinh

In [ ]:
for r in rows[1:]:
    print(f"\n{r['setting']} (n={r['n']})")
    print(f"  accuracy {r['accuracy']:.4f} | macro P/R/F1 {r['macro_precision']:.4f}/{r['macro_recall']:.4f}/{r['macro_f1']:.4f}"
          f" | micro P/R/F1 {r['micro_precision']:.4f}/{r['micro_recall']:.4f}/{r['micro_f1']:.4f}")
    for l in LABELS:
        print(f"  {l:<8} P {r[l + '_precision']:.4f}  R {r[l + '_recall']:.4f}  F1 {r[l + '_f1']:.4f}")
c = cost.iloc[0]
print(f"\nTraining: {c.training_minutes:.1f} min on {c.gpu_training} ({c.train_n} reviews, {c.epochs} epoch)")
print(f"Inference: {c.sec_per_review:.3f} s/review ({c.reviews_per_sec:.1f} reviews/s) on {c.gpu_inference}, batch {c.batch_size}")